#### Read and plot individual timestep from the current folder

More information about LaMEM.jl plotting can be found at https://github.com/JuliaGeodynamics/LaMEM.jl."

In [ ]:
using Plots, LaMEM, GeophysicalModelGenerator, Colors

gr()  # use the GR backend

# Define the current folder and server details
local_folder = pwd()  # Get the current working directory

println("local_folder: ", local_folder)

# Append /output after the current working directory
model_output = joinpath(local_folder, "output")
println("model_output: ", model_output)

local_folder: /Volumes/T7/HR/R_TKIC
model_output: /Volumes/T7/HR/R_TKIC/output


In [ ]:
my_colors = [
    "#FFFFFF",  # 0: air
    "#264e86",  # 1: asthenospheric mantle
    "#6eb6ff",  # 2: oceanic litho mantle
    "#7098da",  # 3: indian litho mantle
    "#7098da",  # 4: asian litho mantle
    "#7098da",  # 5: weakzone box mantle
    "#7098da",  # 6: weakzone slab
    "#7098da",  # 7: weakzone box
    "#7874f2",  # 8: arc litho mantle
    "#2eb872",  # 9: eclogite
    "#2eb872",  # 10: oceanic crust
    "#f0b775",  # 11: weakzone box crust
    "#f0b775",  # 12: asian upper crust
    "#f0b775",  # 13: asian lower crust
    "#f0b775",  # 14: indian sediment
    "#f0b775",  # 15: indian upper crust
    "#f0b775",  # 16: indian lower crust
    "#ef255f",  # 17: arc upper crust
    "#ef255f"   # 18: arc lower crust
]

# #Color scheme with Indian lower crust eclogitization.
# my_colors = [
#     "#FFFFFF",  # 0: air
#     "#264e86",  # 1: asthenospheric mantle
#     "#6eb6ff",  # 2: oceanic litho mantle
#     "#7098da",  # 3: indian litho mantle
#     "#7098da",  # 4: asian litho mantle
#     "#7098da",  # 5: weakzone box mantle
#     "#7098da",  # 6: weakzone slab
#     "#7098da",  # 7: weakzone box
#     "#7874f2",  # 8: arc litho mantle
#     "#2eb872",  # 9: eclogite
#     "#2eb872",  # 10: oceanic crust
#     "#f0b775",  # 11: weakzone box crust
#     "#f0b775",  # 12: asian upper crust
#     "#f0b775",  # 13: asian lower crust
#     "#f0b775",  # 14: indian sediment
#     "#f0b775",  # 15: indian upper crust
#     "#f0b775",  # 16: indian lower crust
#     "#f0b775",  # 17: indian lower crust eclogizitation
#     "#ef255f",  # 18: arc upper crust
#     "#ef255f",  # 19: arc lower crust
#     "#8C5A3C"   # 20: eclogite_crust
# ]

19-element Vector{String}:
 "#FFFFFF"
 "#264e86"
 "#6eb6ff"
 "#7098da"
 "#7098da"
 "#7098da"
 "#7098da"
 "#7098da"
 "#7874f2"
 "#2eb872"
 "#2eb872"
 "#f0b775"
 "#f0b775"
 "#f0b775"
 "#f0b775"
 "#f0b775"
 "#f0b775"
 "#ef255f"
 "#ef255f"

#### Make more plots at this timestep

#### Read all timesteps to generate plots (need to fix the path)

In [ ]:
using Plots, LaMEM, GeophysicalModelGenerator

# Directory for model output and where to save plots
plots_dir =joinpath(local_folder, "plots_phase_vico_density_v2")

# Ensure the plots directory exists
isdir(plots_dir) || mkdir(plots_dir)

# Read all available time steps
all_timesteps, Filenames, time = read_LaMEM_simulation(model_output)
cart_data, _ = read_LaMEM_timestep(model_output,0)

x = cart_data.x.val[:,1,1]
z = cart_data.z.val[1,1,:]

# Define common x and y limits
common_xlims = (-1000, 1000)
common_ylims = (-500, 20)

# Loop through each timestep
for i in 1:length(all_timesteps)
    cart_data, _ = read_LaMEM_timestep(model_output, all_timesteps[i])
    
    phase = (cart_data.fields.phase[:,1,:])'
    temperature = (cart_data.fields.temperature[:,1,:])'
    viscosity= (cart_data.fields.visc_total[:,1,:])'
    j2_strain_rate = (cart_data.fields.j2_strain_rate[:,1,:])'
    density= (cart_data.fields.density[:,1,:])'
    j2_dev_stress= (cart_data.fields.j2_dev_stress[:,1,:])'
    dev_sigma_xx =(cart_data.fields.dev_stress[1][:,1,:])'
    Vx = (cart_data.fields.velocity[1][:,1,:])'
    Vz = (cart_data.fields.velocity[3][:,1,:])'


    p1 = heatmap(x, z, phase,
        title="Phase (ID) @ $(time[i]) Myr", titlefont=10,     color  = cgrad(my_colors; categorical=true),  
        clims  = (-0.5, 18.5), #    clims  = (-0.5, 20.5) with eclogitization
        xlims=common_xlims, ylims=common_ylims, aspect_ratio=1, colorbar=true)

    p2 = heatmap(x, z, temperature, 
        title="Temperature (°C)", titlefont=10, color=cgrad(:roma,rev=true,categorical=true),
        xlims=common_xlims, ylims=common_ylims, aspect_ratio=1, colorbar=true, clims=(20, 1700))

    p3 = heatmap(x, z, viscosity, 
        title="total viscosity (log10,Pas)", titlefont=10, color=cgrad(:vik,rev=true,categorical=true),
        xlims=common_xlims, ylims=common_ylims, aspect_ratio=1, colorbar=true, clims=(19, 23))

    p4 = heatmap(x, z, log10.(j2_strain_rate), 
        title="strain rate (log10, 1/s)", titlefont=10, color=cgrad(:lapaz,rev=false,categorical=true),
        xlims=common_xlims, ylims=common_ylims, aspect_ratio=1, colorbar=true,clims=(-17, -13))

    p5 = heatmap(x, z, density, 
        title="density (kg/m^3)", titlefont=10, color=cgrad(:batlow,rev=true,categorical=true),
        xlims=common_xlims, ylims=common_ylims, aspect_ratio=1, colorbar=true, clims=(2700, 3500))

    p6 = heatmap(x, z, dev_sigma_xx, 
        title="deviatoric sigma_xx (MPa)", titlefont=10, color=cgrad(:RdBu,rev=true,categorical=true),
        xlims=common_xlims, ylims=common_ylims, aspect_ratio=1, colorbar=true,clims=(-300, 300))

    p7 = heatmap(x, z, Vz, title="Vz (cm/yr)",titlefont=10, color=cgrad(:vik,rev=false,categorical=true),
        xlims=(-1000, 1000), ylims=(-500, 20), aspect_ratio=1, colorbar=true, clims=(-20, 20))

    p8 = heatmap(x, z, Vx, title="Vx (cm/yr)",titlefont=10, color=cgrad(:bam,rev=true,categorical=true),
        xlims=(-1000, 1000), ylims=(-500, 20), aspect_ratio=1, colorbar=true, clims=(-20, 20))


    # Combine the plots
    combined_plot = plot(p1, p2, p3, p4, p5, p6, p7, p8, layout=(4,2),size=(1500*0.7,800*0.7), subplot_spacing=1,dpi=300)

    # Format both timestep and counter numbers
    formatted_timestep = lpad(all_timesteps[i], 5, '0') # Pad timestep number to five digits
    formatted_counter = lpad(i, 5, '0')   # Pad counter number to five digits

    # Save the plot with both numbers in the filename
    savefig(combined_plot, "$(plots_dir)/plot_$(formatted_counter)_timestep_$(formatted_timestep).png")
    

end


#### Use ffmpeg to make a movie from image sequence

In [ ]:
# Output video file
output_video_path = joinpath(plots_dir, "output_video.mp4")

# Adjusted ffmpeg command to ignore the timestep part
ffmpeg_command = `ffmpeg -r 30 -pattern_type glob -i $(joinpath(plots_dir, "plot_*.png")) -pix_fmt yuv420p -r 30 $(output_video_path)`

# Run ffmpeg to create the video
run(ffmpeg_command)



ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.4)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1_4 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --e

Process(`ffmpeg -r 30 -pattern_type glob -i '/Volumes/T7/HR/R_TKIC/plots_phase_vico_density_v2/plot_*.png' -pix_fmt yuv420p -r 30 /Volumes/T7/HR/R_TKIC/plots_phase_vico_density_v2/output_video.mp4`, ProcessExited(0))